In [5]:
import pandas as pd
import plotly.express as px

In [19]:
df = pd.read_csv("./experiments/exp1/results.csv")

In [25]:
df.head()

,run,method,p,n,s,seed,mse,shd,precision,recall
0,0,BIC,30,90,0.068966,0,0.005259,17,0.583333,0.913043
1,0,EBIC,30,90,0.068966,0,0.004909,13,0.656250,0.913043
2,1,BIC,30,90,0.068966,1,0.020858,38,0.415094,0.758621
3,1,EBIC,30,90,0.068966,1,0.021122,31,0.478261,0.758621
4,2,BIC,30,90,0.068966,2,0.007484,26,0.574074,0.911765


In [29]:
df['ratio']=df['n']/df['p']


In [36]:
df['k'] = df['s']*(df['p']-1)

In [56]:
df["k_int"] = df["k"].round().astype(int)

In [60]:
df[(df['k_int'] == 2) & (df['p'] == 30)].groupby(by=["method", 'n']).mean()

run     p         s  seed       mse    shd  precision    recall  \
method n                                                                       
BIC    90   49.5  30.0  0.068966  49.5  0.012082  27.84   0.533611  0.855171   
       120  49.5  30.0  0.068966  49.5  0.007155  16.17   0.681459  0.938518   
       150  49.5  30.0  0.068966  49.5  0.002678   9.37   0.784025  0.975118   
EBIC   90   49.5  30.0  0.068966  49.5  0.011746  21.01   0.619762  0.851606   
       120  49.5  30.0  0.068966  49.5  0.006995  12.05   0.751384  0.936243   
       150  49.5  30.0  0.068966  49.5  0.002558   6.58   0.842683  0.975270   

            ratio    k  k_int  
method n                       
BIC    90     3.0  2.0    2.0  
       120    4.0  2.0    2.0  
       150    5.0  2.0    2.0  
EBIC   90     3.0  2.0    2.0  
       120    4.0  2.0    2.0  
       150    5.0  2.0    2.0

In [70]:
from __future__ import annotations
import pandas as pd
import numpy as np
from typing import Optional, Iterable, Literal
import plotly.express as px
import plotly.graph_objects as go

def plot_metric_subplots(
    csv_path: str,
    metric: str,
    *,
    plot_kind: Literal["line", "box"] = "line",
    show_points: bool = False,             # for box: show individual points
    criterion_candidates: Iterable[str] = ("criterion", "penalty", "model", "method", "selector"),
    title: Optional[str] = None,
    aggregate: bool = True,                # ignored for box plots (we’ll force raw)
    width: Optional[int] = None,
    height: Optional[int] = None,
) -> go.Figure:
    """
    Build a faceted Plotly subplot comparing EBIC vs BIC across sample size (n)
    for a given metric, with facets by number of variables (p) and expected parents (k).
    Supports 'line' (means) and 'box' (replicate distributions).

    Parameters
    ----------
    csv_path : str
        Path to the results CSV.
    metric : str
        Metric column to visualize (e.g., 'shd', 'precision', 'recall', 'mse').
    plot_kind : {'line','box'}
        'line' -> mean performance vs n; 'box' -> distribution over replicates per n.
    show_points : bool
        If True (box only), overlay points in each box.
    criterion_candidates : iterable[str]
        Column name candidates that label EBIC/BIC.
    title : str | None
        Custom title; default auto-generated.
    aggregate : bool
        If True (line only), average metric across replicates per (p, k_int, n, criterion).
        Ignored for 'box' (raw data used).
    width, height : int | None
        Optional figure size in pixels.

    Returns
    -------
    plotly.graph_objects.Figure
    """
    # Load
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]

    # Required columns
    needed_base = {"p", "n"}
    if not needed_base.issubset(df.columns):
        raise ValueError(f"CSV must contain columns {needed_base}, got {set(df.columns)}.")

    # Derived columns
    if "ratio" not in df.columns:
        df["ratio"] = df["n"] / df["p"]
    if "k" not in df.columns and "s" in df.columns:
        df["k"] = df["s"] * (df["p"] - 1)
    if "k" in df.columns and "k_int" not in df.columns:
        df["k_int"] = df["k"].round().astype(int)

    # Detect EBIC/BIC column
    criterion_col = next((c for c in criterion_candidates if c in df.columns), None)
    if criterion_col is None:
        raise ValueError(
            "Could not find a column that identifies EBIC vs BIC. "
            f"Tried: {criterion_candidates}. "
            "Ensure your CSV has a column indicating the selection criterion."
        )

    # Check metric column
    if metric not in df.columns:
        raise ValueError(f"Metric '{metric}' not found in CSV. Columns: {list(df.columns)}")

    # Need k_int for faceting
    if "k_int" not in df.columns:
        if "k" in df.columns and np.issubdtype(df["k"].dtype, np.number):
            df["k_int"] = df["k"].round().astype(int)
        else:
            raise ValueError(
                "Neither 'k_int' nor a numeric 'k' is available. "
                "Provide 's' (so k = s*(p-1)) or include 'k' in the CSV."
            )

    # Tidy table
    tidy = df[["p", "k_int", "n", criterion_col, metric]].dropna().copy()

    fig_title = title or (
        f"{metric.upper()} vs Sample Size (n) — EBIC vs BIC (faceted by p and k) "
        + ("[box]" if plot_kind == "box" else "")
    )

    if plot_kind == "line":
        # Aggregate if requested
        data = (
            tidy.groupby(["p", "k_int", "n", criterion_col], as_index=False)
                .agg({metric: "mean"})
            if aggregate else tidy
        )
        data = data.sort_values(["p", "k_int", "n", criterion_col])
        fig = px.line(
            data,
            x="n",
            y=metric,
            color=criterion_col,
            facet_row="p",
            facet_col="k_int",
            markers=True,
            title=fig_title,
        )
        fig.update_yaxes(matches="y")
        fig.update_xaxes(matches=None)
    elif plot_kind == "box":
        # Box plots use raw replicate data (ignore aggregate)
        fig = px.box(
            tidy,
            x="n",
            y=metric,
            color=criterion_col,
            facet_row="p",
            facet_col="k_int",
            points="all" if show_points else False,  # 'all' | 'outliers' | False
            title=fig_title,
        )
        # Helpful for large n values (treat x as categorical bins)
        # fig.update_xaxes(type="category")
        fig.update_yaxes(matches="y")
        fig.update_xaxes(matches=None)
    else:
        raise ValueError("plot_kind must be 'line' or 'box'.")

    fig.update_layout(legend_title_text="Criterion", hovermode="x unified")

    # Optional sizing
    if width is not None or height is not None:
        fig.update_layout(width=width, height=height)

    return fig


In [81]:
from plotly.io import show

fig = plot_metric_subplots("./experiments/exp2/results.csv", metric="shd", plot_kind="line")
show(fig)  

In [80]:
fig.write_html("mse_box.html")

In [82]:
metrics = ["shd","precision","recall","mse"]
plt_type =["line", "box"]
chart_prefix = "exp_2"
for m in metrics:
    for plt_t in plt_type:
        fig = plot_metric_subplots("./experiments/exp2/results.csv", metric=m, plot_kind=plt_t)
        fig.write_html(f"{chart_prefix}_{m}_{plt_t}.html")